# Four Dots That Look the Same (CODA-CS)

**Paper Companion:** Section 4 (Theorems CS-T3, CS-T4) & Section 5 (Theorem CS-T5).

This notebook explores the core obstructions in CODA-CS:
1. **Theorem CS-T3 (First Coordinate Does Not Determine Second):**
   $$\neg \mathsf{Factors}(x \mapsto x.1, x \mapsto x.2).$$
2. **Theorem CS-T4 (Fine Factors When Coarse Cannot):**
   $$\mathsf{Factors}(x \mapsto x, x \mapsto x.2) \land \neg \mathsf{Factors}(x \mapsto x.1, x \mapsto x.2).$$
3. **Theorem CS-T5 (Continuum Limit Does Not Create Authority):** Missing terminalizer blocks query authority; limit notation does not manufacture authority.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_cs_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_cs_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_cs_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Matrix, Rational
from coda_cs_core import (
    ContinuumQuery,
    ContinuumSignal,
    ProfileMap,
    ShadowSlice,
    ShadowTower,
    evaluate_query_authority,
)
print("Dependencies loaded.")

Dependencies loaded.


## 1. The Collision: Why Factorization Fails (Theorem CS-T3)

Consider the four states in $X = \mathbf{2} \times \mathbf{2}$:

| State $x$ | First Coordinate $x.1$ | Second Coordinate $x.2$ |
| :--- | :---: | :---: |
| $(\mathsf{false}, \mathsf{false})$ | $\mathsf{false}$ | $\mathsf{false}$ |
| $(\mathsf{false}, \mathsf{true})$ | $\mathsf{false}$ | $\mathsf{true}$ |
| $(\mathsf{true}, \mathsf{false})$ | $\mathsf{true}$ | $\mathsf{false}$ |
| $(\mathsf{true}, \mathsf{true})$ | $\mathsf{true}$ | $\mathsf{true}$ |

Suppose $q(x) = x.2$ factored through $p(x) = x.1$. Then there would exist a function $g: \mathbf{2} \to \mathbf{2}$ such that:
$$g(p(x)) = q(x) \quad \forall x.$$
Evaluating on the first two rows:
- Row 1: $g(\mathsf{false}) = \mathsf{false}$.
- Row 2: $g(\mathsf{false}) = \mathsf{true}$.

This implies $\mathsf{false} = \mathsf{true}$, a direct contradiction! Thus, $\neg \mathsf{Factors}(x \mapsto x.1, x \mapsto x.2)$.

In [2]:
pairs = [(False, False), (False, True), (True, False), (True, True)]
first = lambda x: x[0]
second = lambda x: x[1]
identity = lambda x: x

def factors(p, q, domain):
    table = {}
    for x in domain:
        px, qx = p(x), q(x)
        if px in table and table[px] != qx:
            return False
        table[px] = qx
    return True

f_first = factors(first, second, pairs)
f_id = factors(identity, second, pairs)

print(f"Does second coordinate factor through first coordinate? {f_first}")
print(f"Does second coordinate factor through identity?         {f_id}")

assert f_first is False  # Theorem CS-T3
assert f_id is True     # Theorem CS-T4
print("VERIFIED: Theorems CS-T3 and CS-T4 successfully demonstrated.")

Does second coordinate factor through first coordinate? False
Does second coordinate factor through identity?         True
VERIFIED: Theorems CS-T3 and CS-T4 successfully demonstrated.


## 2. Limit Notation is Not an Answer (Theorem CS-T5)

Theorem CS-T5 establishes that writing a continuum limit or name does NOT manufacture authority. In the computational engine, authority is granted **query-relatively** only when all 8 admissibility gates pass.

Here we test the mechanical firewall across the 4 canonical test cases:

In [3]:
pts = (Rational(0, 4), Rational(1, 4), Rational(2, 4), Rational(3, 4))
prof = ProfileMap(profile_id="Grid4", sample_points=pts)
sig = ContinuumSignal("LinearSig", {p: Rational(p) for p in pts})

q_good = ContinuumQuery("Energy", degree_bound=2, evaluator=lambda s: Rational(1, 2))
tower_good = ShadowTower("GoodTower", slices=(ShadowSlice("S1", 4, True),), is_cofinal=True)

# 1. Admissible case
res_admissible = evaluate_query_authority(
    signal=sig,
    query=q_good,
    tower=tower_good,
    profile=prof,
    factored_evaluator=lambda v: Rational(1, 2),
    d0_transition=Matrix([[1]]),
    defect_1cochain=Matrix([[0]]),
)

# 2. Missing terminalizer (Theorem CS-T5 witness)
res_no_term = evaluate_query_authority(
    signal=sig,
    query=q_good,
    tower=tower_good,
    profile=prof,
    factored_evaluator=lambda v: Rational(1, 2),
    has_terminalizer=False,
    d0_transition=Matrix([[1]]),
    defect_1cochain=Matrix([[0]]),
)

# 3. Non-cofinal tower
tower_bad = ShadowTower("BadTower", slices=(ShadowSlice("S1", 4, True),), is_cofinal=False)
res_non_cofinal = evaluate_query_authority(
    signal=sig,
    query=q_good,
    tower=tower_bad,
    profile=prof,
    factored_evaluator=lambda v: Rational(1, 2),
)

# 4. Unbounded degree
q_high = ContinuumQuery("HighDeg", degree_bound=10, evaluator=lambda s: Rational(1, 2))
res_unbounded = evaluate_query_authority(
    signal=sig,
    query=q_high,
    tower=tower_good,
    profile=prof,
    factored_evaluator=lambda v: Rational(1, 2),
)

print(f"{'Case':<25} | {'Inhabited?':<12} | {'Residual Mass':<15} | {'Refusal Code'}")
print("-" * 70)
cases = [
    ("Admissible Query", res_admissible),
    ("Missing Terminalizer", res_no_term),
    ("Non-Cofinal Tower", res_non_cofinal),
    ("Unbounded Degree", res_unbounded),
]
for name, r in cases:
    print(f"{name:<25} | {str(r.inhabited):<12} | {str(r.residual_mass):<15} | {r.refusal_code}")

assert res_admissible.inhabited is True
assert res_admissible.residual_mass == Rational(0, 1)
assert res_no_term.inhabited is False
assert res_no_term.refusal_code == "NoTerminal"
assert res_non_cofinal.inhabited is False
assert res_non_cofinal.refusal_code == "NoCofinalAuthorizedShadow"
assert res_unbounded.inhabited is False
assert res_unbounded.refusal_code == "R_unbounded_degree"

print("-" * 70)
print("VERIFIED: The firewall strictly blocks authority without valid descent certificates.")

Case                      | Inhabited?   | Residual Mass   | Refusal Code
----------------------------------------------------------------------
Admissible Query          | True         | 0               | None
Missing Terminalizer      | False        | 1/4             | NoTerminal
Non-Cofinal Tower         | False        | 1               | NoCofinalAuthorizedShadow
Unbounded Degree          | False        | 1               | R_unbounded_degree
----------------------------------------------------------------------
VERIFIED: The firewall strictly blocks authority without valid descent certificates.
